## 실험 2. 트리 모델의 Sharpe Ratio 비교

**목차**
1. 데이터 X/y 분리 + train/val 분할 <br>
2. 통제칸용 47열 데이터 준비 <br>
3. 트리 × 47열 학습 (실험②) <br>
4. 트리 × 122열 학습 (실험③) <br>
5. val Sharpe 최적화 <br>
6. 하이퍼파라미터 튜닝<br>
7. 성능 평가

### 0. 데이터 로드

In [1]:
import pandas as pd

# 트리 모델용 데이터 로드 (122열 + y + id)
df_tree = pd.read_csv("model_input_tree.csv")

print("전체 shape:", df_tree.shape)                          # (행 개수, 열 개수) 확인
print("split 컬럼 분포:\n", df_tree['split'].value_counts())  # train/val/test 개수 확인

전체 shape: (717969, 133)
split 컬럼 분포:
 split
train    430781
test     143595
val      143593
Name: count, dtype: int64


### 1. X / y 분리 + train / val 분할

- `df_tree`(133열)를 X(feature 122개) / y(타깃 3종) / 메타데이터로 분리
- `split` 컬럼 기준으로 train / val 필터링

**누수 주의**: feature로 쓰면 안 되는 열 11개 `id, split, grade, term_n, funded_amnt, rf, irr, excess, excess_contract, spread_positive, bad`
- `excess`, `irr`, `rf`: y를 만드는 재료라 X에 넣으면 정답을 미리 보는 것
- `grade`: LC가 이미 내린 판단이라 넣으면 모델이 LC를 복사함. 다만 등급대별 배분에 쓰이므로 따로 보관

#### 1-1. 금지 컬럼 정의 및 존재 여부 확인

In [2]:
# feature로 쓰면 안 되는 열 11개
FORBIDDEN = [
    'id',               # 조인 키 (식별자, 학습 의미 없음)
    'split',            # train/val/test 구분용 (학습 입력 아님)
    'grade',            # LC 등급 — 학습 금지, 포트폴리오 배분에만 사용
    'term_n',           # 대출 기간 (사후 정보 성격)
    'funded_amnt',      # 실제 펀딩 금액 (사후 정보 성격)
    'rf',               # 무위험수익률 (y 계산 재료)
    'irr',              # 내부수익률 (y 계산 재료)
    'excess',           # 초과수익 = irr - rf → 주 타깃(y), 절대 X에 넣으면 안 됨
    'excess_contract',  # 계약 기준 초과수익 (y 변형)
    'spread_positive',  # 초과수익 양수 여부 → 보조 타깃(y)
    'bad',              # 부도 여부 → 진단용 타깃(y)
]

# 실제 df_tree에 이 컬럼들이 있는지 사전 확인
# → 혹시 컬럼명이 다르면 여기서 바로 잡힘
missing = [col for col in FORBIDDEN if col not in df_tree.columns]

if missing:
    print(f"⚠️ df_tree에 없는 금지 컬럼 (이름 확인 필요): {missing}")
else:
    print("✅ 금지 컬럼 11개 전부 데이터에 존재 확인 완료")

✅ 금지 컬럼 11개 전부 데이터에 존재 확인 완료


In [3]:
# X: 금지 컬럼 11개를 전부 제거한 나머지 = 순수 feature
X_all = df_tree.drop(columns=FORBIDDEN)

# y: 타깃 3종 분리
# excess(연속) vs spread_positive(이진) 둘 다 실험해서 val 샤프로 결정
y_excess = df_tree['excess']           # 주 타깃: 국채 대비 초과수익 (연속값)
y_binary = df_tree['spread_positive']  # 보조 타깃: 초과수익 > 0 이면 1 (이진)
y_bad    = df_tree['bad']              # 진단용: 부도 여부 (직접 학습 타깃으로 쓰지 않음)

# 메타데이터: 학습엔 안 쓰지만 나중에 필요한 컬럼들
# grade → 챕터 8 포트폴리오 등급대별 배분 시 사용
meta = df_tree[['id', 'split', 'grade']].copy()

print(f"X_all shape: {X_all.shape}")   # (717969, 122) 예상
print(f"y_excess shape: {y_excess.shape}")
print(f"메타데이터 컬럼: {meta.columns.tolist()}")

X_all shape: (717969, 122)
y_excess shape: (717969,)
메타데이터 컬럼: ['id', 'split', 'grade']


#### 1-2. 안전장치: X에 금지 컬럼이 없는지 최종 확인

In [4]:
# X에 금지 컬럼이 단 하나라도 섞여 있으면 즉시 에러 발생
# → 에러가 안 나면 안전하다는 뜻
assert not any(col in X_all.columns for col in FORBIDDEN), \
    "🚨 금지 feature가 X에 섞여 있음! 즉시 확인 필요!"

print("✅ 안전장치 통과 — X에 금지 feature 없음")
print(f"X 최종 컬럼 수: {X_all.shape[1]}")  # 122여야 함 (133 - 11 = 122)

✅ 안전장치 통과 — X에 금지 feature 없음
X 최종 컬럼 수: 122


#### 1-3. train / val 필터링

- `split` 컬럼 기준으로 필터링 (이미 정해진 분할을 따름)
- test는 절대 건드리지 않음: 챔피언 모델 확정 후 딱 1회만 사용

In [5]:
# split 컬럼 기준으로 train/val 행만 골라내는 불리언 마스크 생성
train_mask = (meta['split'] == 'train')  # train 행: True
val_mask   = (meta['split'] == 'val')    # val 행: True
# test_mask는 지금 만들지 않음 — 챔피언 확정 전까지 손대지 않는 원칙

# X 분리
X_train = X_all[train_mask].reset_index(drop=True)  # train feature
X_val   = X_all[val_mask].reset_index(drop=True)    # val feature

# y 분리 (타깃 3종 × train/val)
y_train_excess = y_excess[train_mask].reset_index(drop=True)
y_val_excess   = y_excess[val_mask].reset_index(drop=True)

y_train_binary = y_binary[train_mask].reset_index(drop=True)
y_val_binary   = y_binary[val_mask].reset_index(drop=True)

y_train_bad    = y_bad[train_mask].reset_index(drop=True)
y_val_bad      = y_bad[val_mask].reset_index(drop=True)

# 메타데이터 분리 (나중에 grade 기반 포트폴리오 배분 등에 사용)
meta_train = meta[train_mask].reset_index(drop=True)
meta_val   = meta[val_mask].reset_index(drop=True)

print("===== shape 확인 =====")
print(f"X_train: {X_train.shape}")  # (430781, 122) 예상
print(f"X_val:   {X_val.shape}")    # (143593, 122) 예상

===== shape 확인 =====
X_train: (430781, 122)
X_val:   (143593, 122)


#### 1-4. 최종 검증

In [6]:
# 문서에 명시된 숫자와 실제 shape이 일치하는지 assert로 확인
assert X_train.shape == (430781, 122), f"🚨 X_train shape 불일치: {X_train.shape}"
assert X_val.shape   == (143593, 122), f"🚨 X_val shape 불일치: {X_val.shape}"

print("✅ 모든 shape 문서 숫자와 일치")
print()
print(f"y_train_excess 기초 통계:\n{y_train_excess.describe()}")
print()
print(f"y_train_binary 분포 (0/1 비율):\n{y_train_binary.value_counts(normalize=True).round(4)}")
# spread_positive 양성 비율: 문서에 85.44%라고 명시됨 → 확인용

✅ 모든 shape 문서 숫자와 일치

y_train_excess 기초 통계:
count    430781.000000
mean          0.022947
std           0.267066
min          -1.022579
25%           0.062859
50%           0.104809
75%           0.139730
max           0.993148
Name: excess, dtype: float64

y_train_binary 분포 (0/1 비율):
spread_positive
1    0.8544
0    0.1456
Name: proportion, dtype: float64


### 2. 통제칸용 47열 데이터 준비 (실험②용)

트리 모델을 122열로만 돌리면 '선형 모델보다 트리 모델이 우수하다'는 결과가 모델 구조 덕분인지, 변수를 75개 더 받아서인지 구분이 안 됨. 같은 변수(47열) 조건에서 비교하는 통제칸이 필요하다.

#### 2-1. 공통변수 목록 로드

In [ ]:
linear_cols = pd.read_csv("model_input_linear.csv", nrows=0).columns.tolist()
common_cols = [c for c in linear_cols if c in X_train.columns]

print(f"공통변수 개수: {len(common_cols)}")
print(f"앞 10개: {common_cols[:10]}")

공통변수 개수: 47
앞 10개: ['loan_amnt', 'term', 'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc']


#### 2-2. X_train / X_val 에서 47열만 골라내기

In [8]:
# 공통변수 47개가 실제로 X_train 안에 전부 있는지 먼저 확인
missing_cols = [col for col in common_cols if col not in X_train.columns]

if missing_cols:
    print(f"⚠️ X_train에 없는 공통변수: {missing_cols}")
else:
    print("✅ 공통변수 47개 전부 X_train 안에 존재 확인")

# 47열만 골라내서 통제칸용 데이터 생성
# X_train_ctrl → 실험② (트리 × 47열) 에 사용
# X_val_ctrl   → 실험② val 평가에 사용
X_train_ctrl = X_train[common_cols]  # train: 47열만
X_val_ctrl   = X_val[common_cols]    # val: 47열만

print(f"\nX_train_ctrl shape: {X_train_ctrl.shape}")  # (430781, 47) 예상
print(f"X_val_ctrl shape:   {X_val_ctrl.shape}")      # (143593, 47) 예상

✅ 공통변수 47개 전부 X_train 안에 존재 확인

X_train_ctrl shape: (430781, 47)
X_val_ctrl shape:   (143593, 47)


#### 2-3. 최종 검증

In [9]:
# shape 확인
assert X_train_ctrl.shape == (430781, 47), f"🚨 X_train_ctrl shape 불일치: {X_train_ctrl.shape}"
assert X_val_ctrl.shape   == (143593, 47), f"🚨 X_val_ctrl shape 불일치: {X_val_ctrl.shape}"

# 선형용 47열이 트리용 122열의 부분집합인지 확인 (인계 문서: 47열 ⊂ 122열)
assert set(common_cols).issubset(set(X_train.columns)), \
    "🚨 공통변수가 X_train 122열의 부분집합이 아님!"

print("✅ 통제칸 데이터 준비 완료")
print()
print("===== 지금까지 만들어진 데이터 정리 =====")
print(f"실험② 트리×47  → X_train_ctrl: {X_train_ctrl.shape} / X_val_ctrl: {X_val_ctrl.shape}")
print(f"실험③ 트리×122 → X_train:      {X_train.shape}      / X_val:      {X_val.shape}")

✅ 통제칸 데이터 준비 완료

===== 지금까지 만들어진 데이터 정리 =====
실험② 트리×47  → X_train_ctrl: (430781, 47) / X_val_ctrl: (143593, 47)
실험③ 트리×122 → X_train:      (430781, 122)      / X_val:      (143593, 122)


### 3. 트리 × 47열 학습 (실험②)

- LightGBM, XGBoost, CatBoost 3개 모델을 47열(통제칸)로 학습 (RF는 튜닝 대상 아니라 제외)
- 타깃: `excess`(연속)
- 해당 모델에 대한 평가(샤프 계산)는 6절에서 진행한다.

#### 3-1. 라이브러리 설치 및 임포트

In [ ]:
# 필요한 라이브러리 설치 (Colab 환경에서 없을 경우 대비)
!pip install lightgbm xgboost catboost -q  # -q: 설치 로그 최소화

# 트리 모델 3종 임포트 (RF는 튜닝 대상 아니라 미사용)
from xgboost import XGBRegressor, XGBClassifier                            # XGBoost
from lightgbm import LGBMRegressor, LGBMClassifier                         # LightGBM
from catboost import CatBoostRegressor, CatBoostClassifier                 # CatBoost

import numpy as np  # 수치 계산
import time         # 학습 시간 측정용

print("✅ 라이브러리 임포트 완료")

#### 3-2. 모델 정의

| 모델 | 특징 |
|---|---|
| LightGBM | 가장 빠름, NaN 자체 처리 가능 |
| XGBoost | 범용적, NaN 자체 처리 가능 |
| CatBoost | 범주형 변수에 강함, 별도 인코딩 불필요 |

- RF는 6절 분석 결과 노이즈에 취약하고 튜닝 대상도 아니므로 학습에서 제외
- 타깃은 excess(연속)만 사용 — 예측값 부호로 이진 판단도 가능(양수=좋은 대출)
- `n_jobs=4`, `random_state=42` 고정

In [ ]:
# ── excess(연속) 타깃용 모델 3종 (RF 제외) ──────────────────
# → 연속 모델 예측값으로 이진 판단 가능 (양수=좋은 대출, 음수=나쁜 대출)

models_reg = {
    "LGBM": LGBMRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                n_jobs=4,
                random_state=42,
                verbosity=-1        # 학습 로그 끄기
            ),
    "XGB":  XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                n_jobs=4,
                random_state=42,
                verbosity=0         # 학습 로그 끄기
            ),
    "CAT":  CatBoostRegressor(
                iterations=300,     # CatBoost는 n_estimators 대신 iterations
                depth=6,            # CatBoost는 max_depth 대신 depth
                learning_rate=0.05,
                random_seed=42,     # CatBoost는 random_state 대신 random_seed
                verbose=0           # 학습 로그 끄기
            ),
}

print("✅ 모델 정의 완료 (연속 타깃만, RF 제외 부스팅 3종)")
print(f"  모델: {list(models_reg.keys())}")

#### 3-3. 학습: 트리 × 47열 (통제칸)

In [ ]:
import pickle

# 로컬 실행: 세션이 끊겨도 이어할 수 있도록 모델별 체크포인트 저장/로드
PREDS_CTRL_PATH = "preds_ctrl.pkl"

if os.path.exists(PREDS_CTRL_PATH):
    with open(PREDS_CTRL_PATH, "rb") as f:
        preds_ctrl = pickle.load(f)
    print(f"✅ 기존 체크포인트 로드 — 이미 완료된 모델: {list(preds_ctrl['reg'].keys())}")
else:
    preds_ctrl = {"reg": {}}
    print("✅ 새 preds_ctrl 생성 (체크포인트 없음)")

print("===== 실험② 트리 × 47열 (통제칸) 학습 시작 =====\n")

# ── 연속 타깃(excess) ─────────────────────────────────────
print("[ 연속 타깃: excess ]")
for name, model in models_reg.items():
    if name in preds_ctrl["reg"]:
        print(f"  ⏭ {name} — 이미 완료됨, 건너뜀")
        continue
    start = time.time()
    model.fit(X_train_ctrl, y_train_excess)              # train으로만 학습
    elapsed = time.time() - start

    preds_ctrl["reg"][name] = model.predict(X_val_ctrl)  # val 예측값 저장
    with open(PREDS_CTRL_PATH, "wb") as f:
        pickle.dump(preds_ctrl, f)
    print(f"  ✅ {name} 완료 ({elapsed:.0f}초) — 체크포인트 저장됨")

print("\n✅ 실험② 전체 학습 완료")
print("\n[ 저장된 예측값 확인 ]")
for name, pred in preds_ctrl["reg"].items():
    print(f"  preds_ctrl['reg']['{name}']: shape {pred.shape}")

### 4. 트리 × 122열 학습 (실험③)

- LightGBM / XGBoost / CatBoost 3개 모델을 122열 전체로 학습 
- 타깃: `excess`(연속)
- 나머지 조건은 실험②와 동일 (같은 val 행, 같은 하이퍼파라미터)

#### 4-1. 모델 정의 

In [ ]:
# ── excess(연속) 타깃용 모델 3종 ──────────────────
models_reg_full = {
    "LGBM": LGBMRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                n_jobs=4,
                random_state=42,
                verbosity=-1        # 학습 로그 끄기
            ),
    "XGB":  XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                n_jobs=4,
                random_state=42,
                verbosity=0         # 학습 로그 끄기
            ),
    "CAT":  CatBoostRegressor(
                iterations=300,     # CatBoost는 n_estimators 대신 iterations
                depth=6,            # CatBoost는 max_depth 대신 depth
                learning_rate=0.05,
                random_seed=42,     # CatBoost는 random_state 대신 random_seed
                verbose=0           # 학습 로그 끄기
            ),
}

print("✅ 실험③ 모델 정의 완료 (연속 타깃만, RF 제외 부스팅 3종)")
print(f"  모델: {list(models_reg_full.keys())}")

#### 4-2. 학습: 트리 × 122열

In [ ]:
import pickle

# 로컬 실행: 세션이 끊겨도 이어할 수 있도록 모델별 체크포인트 저장/로드
PREDS_FULL_PATH = "preds_full.pkl"

if os.path.exists(PREDS_FULL_PATH):
    with open(PREDS_FULL_PATH, "rb") as f:
        preds_full = pickle.load(f)
    print(f"✅ 기존 체크포인트 로드 — 이미 완료된 모델: {list(preds_full['reg'].keys())}")
else:
    preds_full = {"reg": {}}
    print("✅ 새 preds_full 생성 (체크포인트 없음)")

print("===== 실험③ 트리 × 122열 학습 시작 =====\n")

print("[ 연속 타깃: excess ]")
for name, model in models_reg_full.items():
    if name in preds_full["reg"]:
        print(f"  ⏭ {name} — 이미 완료됨, 건너뜀")
        continue
    start = time.time()
    model.fit(X_train, y_train_excess)           # 122열 전체로 학습
    elapsed = time.time() - start

    preds_full["reg"][name] = model.predict(X_val)  # val 예측값 저장
    with open(PREDS_FULL_PATH, "wb") as f:
        pickle.dump(preds_full, f)
    print(f"  ✅ {name} 완료 ({elapsed:.0f}초) — 체크포인트 저장됨")

print("\n✅ 실험③ 전체 학습 완료")
print("\n[ 저장된 예측값 확인 ]")
for name, pred in preds_full["reg"].items():
    print(f"  preds_full['reg']['{name}']: shape {pred.shape}")

### 5. val Sharpe 최적화

1. pct 그리드서치: 모든 조합에 대해 승인률을 바꿔가며 Sharpe 최댓값 탐색
2. 실험② vs 실험③ 비교: 47열 vs 122열 중 어느 쪽이 Sharpe 더 높은지
* 세 실험 모두 동일한 val 행을 사용.

벤치마크 및 비교 대상: <br>

| 기준 | 샤프 |
|---|---|
| 전건 거부 (전액 국채) | 0.0000 |
| 전건 승인 | 0.0874 |
| A·B등급만 승인 (벤치마크) | 0.1297 |
| LASSO 참고치 | 0.1675 |
| Oracle 상한 (목표 아님) | 1.6246 |

#### 5-1. pct 그리드서치 함수 정의

In [ ]:
def sharpe(pred, excess, pct):
    approve = pred >= np.quantile(pred, 1 - pct)  # 승인 기준 임계값
    x = np.where(approve, excess, 0.0)            # 거부 건은 0
    return x.mean() / x.std(ddof=1)              # 동일가중 샤프


def grid_search_pct(pred, excess, pct_range=None):
    """
    pct 그리드서치 — 샤프 최대화하는 승인률 탐색

    Parameters:
        pred      : 모델 val 예측값
        excess    : val 실제 초과수익 (y_val_excess)
        pct_range : 탐색할 pct 범위 (기본: 0.05~0.95)

    Returns:
        best_pct    : 샤프 최대인 승인률
        best_sharpe : 최대 샤프값
        results     : 전체 pct별 샤프 딕셔너리
    """
    if pct_range is None:
        pct_range = np.arange(0.05, 1.00, 0.05)  # 5%~95%, 5% 단위

    results = {}
    for pct in pct_range:
        results[round(pct, 2)] = sharpe(pred, excess, pct)

    best_pct    = max(results, key=results.get)  # 샤프 최대인 pct
    best_sharpe = results[best_pct]              # 최대 샤프값

    return best_pct, best_sharpe, results

print("✅ 샤프 함수 및 그리드서치 함수 정의 완료")

#### 5-2. 전체 조합 Sharpe 계산

In [ ]:
# val 실제 초과수익 (numpy array로 변환)
excess_val = y_val_excess.to_numpy()

# 전체 결과 저장
all_results = []

# ── 실험② (트리 × 47열) ───────────────────────────────────
for model_name, pred in preds_ctrl["reg"].items():
    best_pct, best_sharpe, _ = grid_search_pct(pred, excess_val)
    all_results.append({
        "실험":    "② 트리×47",
        "모델":    model_name,
        "최적pct": best_pct,
        "최고샤프": round(best_sharpe, 4)
    })

# ── 실험③ (트리 × 122열) ──────────────────────────────────
for model_name, pred in preds_full["reg"].items():
    best_pct, best_sharpe, _ = grid_search_pct(pred, excess_val)
    all_results.append({
        "실험":    "③ 트리×122",
        "모델":    model_name,
        "최적pct": best_pct,
        "최고샤프": round(best_sharpe, 4)
    })

# 데이터프레임으로 정리 후 샤프 기준 내림차순 정렬
df_results = pd.DataFrame(all_results).sort_values("최고샤프", ascending=False)
df_results = df_results.reset_index(drop=True)

print("===== 전체 조합 val 샤프 결과 (내림차순) =====")
print(df_results.to_string(index=True))
print()
print(f"벤치마크 (A·B등급만 승인): 0.1297")
print(f"벤치마크 초과 조합 수: {(df_results['최고샤프'] > 0.1297).sum()} / {len(df_results)}")

#### 5-2 완료: 전체 조합 val 샤프 결과 (각 모델 최적 pct 기준)

| 순위 | 실험 | 모델 | 최적pct | 최고샤프 |
|---|---|---|---|---|
| 1 | ③ 트리×122 | LGBM | 0.40 | 0.1931 |
| 2 | ③ 트리×122 | XGB | 0.45 | 0.1911 |
| 3 | ③ 트리×122 | CAT | 0.40 | 0.1908 |
| 4 | ② 트리×47 | CAT | 0.45 | 0.1889 |
| 5 | ② 트리×47 | LGBM | 0.45 | 0.1871 |
| 6 | ② 트리×47 | XGB | 0.45 | 0.1859 |
| 7 | ② 트리×47 | RF | 0.45 | 0.1690 |
| 8 | ③ 트리×122 | RF | 0.45 | 0.1665 |

벤치마크(A·B등급만 승인) 0.1297 대비 8개 조합 전부 초과. <br>


**💡 인사이트**
- 모든 조합이 벤치마크를 초과: "LC 등급이 개별 편차를 뭉갠다"는 핵심 주장을 뒷받침
- LGBM×122(pct=0.40), CAT×122(pct=0.40)는 최적 pct가 서로 달라, 변수 효과(③−②)를 순수하게 비교하려면 pct=0.45(실험② 기준)로 통일해 재계산 필요
- RF는 47열(0.1690)이 122열(0.1665)보다 높음: 추가 변수가 오히려 노이즈로 작용. 부스팅 계열은 NaN을 분기 조건으로 활용하지만 RF는 고결측 변수(sec_app_* 93%)를 효과적으로 처리하지 못한 것으로 해석. 이후 분석에서 RF 제외, 부스팅 3종만 사용
- 다음 단계: pct=0.45로 통일해 부스팅 3종 × 47/122열 총 6개 조합 튜닝

In [ ]:
# pct=0.45 기준 전체 결과표 정리
# (원본 셀은 정의되지 않은 MODELS/results_045를 참조하는 버그가 있어 preds_ctrl/preds_full로 재작성)
all_results_045 = []

for model_name, pred in preds_ctrl["reg"].items():
    all_results_045.append({
        "실험": "② 트리×47",
        "모델": model_name,
        "pct": 0.45,
        "샤프": round(sharpe(pred, excess_val, 0.45), 4)
    })

for model_name, pred in preds_full["reg"].items():
    all_results_045.append({
        "실험": "③ 트리×122",
        "모델": model_name,
        "pct": 0.45,
        "샤프": round(sharpe(pred, excess_val, 0.45), 4)
    })

df_results_045 = pd.DataFrame(all_results_045).sort_values("샤프", ascending=False)
df_results_045 = df_results_045.reset_index(drop=True)

print("===== 전체 조합 val 샤프 결과 (pct=0.45 통일) =====")
print(df_results_045.to_string(index=True))
print()
print(f"벤치마크 (A·B등급만 승인): 0.1297")
print(f"벤치마크 초과 조합 수: {(df_results_045['샤프'] > 0.1297).sum()} / {len(df_results_045)}")

### 6. 하이퍼파라미터 튜닝
앞선 3·4절의 개별 학습은 방향성 확인용. 최종 확정 파라미터는 본 절의 튜닝 결과를 따른다.

- 부스팅 3종(LGBM/XGB/CAT) × 47열/122열 = 6개 조합을 Optuna(seed=42)로 각각 튜닝
- pct는 0.45로 고정(실험②의 최적값 기준 통일): 변수 효과(③−②)를 pct 차이 없이 순수하게 비교하기 위함

#### 6-1. Optuna 설치 및 임포트

In [ ]:
!pip install optuna -q  # Optuna 설치

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # 로그 최소화

print("✅ Optuna 임포트 완료")

#### 6-2. 튜닝 함수 정의

In [ ]:
excess_val = y_val_excess.to_numpy()  # val 실제 초과수익

def tune_lgbm(X_train_data, X_val_data, best_pct, n_trials=50):
    """LGBM 하이퍼파라미터 튜닝 (pct 고정, seed=42)"""
    def objective(trial):
        params = {
            "n_estimators":  trial.suggest_int("n_estimators", 100, 1000),
            "max_depth":     trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "num_leaves":    trial.suggest_int("num_leaves", 20, 150),
            "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
            "n_jobs": 4,
            "random_state": 42,
            "verbosity": -1
        }
        model = LGBMRegressor(**params)
        model.fit(X_train_data, y_train_excess)   # train으로만 학습
        pred = model.predict(X_val_data)          # val 예측
        return sharpe(pred, excess_val, best_pct) # val 샤프 최대화 (pct 고정)

    sampler = optuna.samplers.TPESampler(seed=42) # 재현성 고정
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value


def tune_xgb(X_train_data, X_val_data, best_pct, n_trials=50):
    """XGB 하이퍼파라미터 튜닝 (pct 고정, seed=42)"""
    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 1000),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "n_jobs": 4,
            "random_state": 42,
            "verbosity": 0
        }
        model = XGBRegressor(**params)
        model.fit(X_train_data, y_train_excess)
        pred = model.predict(X_val_data)
        return sharpe(pred, excess_val, best_pct) # val 샤프 최대화 (pct 고정)

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value


def tune_cat(X_train_data, X_val_data, best_pct, n_trials=50):
    """CAT 하이퍼파라미터 튜닝 (pct 고정, seed=42)"""
    def objective(trial):
        params = {
            "iterations":    trial.suggest_int("iterations", 100, 1000),
            "depth":         trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "random_seed": 42,
            "verbose": 0
        }
        model = CatBoostRegressor(**params)
        model.fit(X_train_data, y_train_excess)
        pred = model.predict(X_val_data)
        return sharpe(pred, excess_val, best_pct) # val 샤프 최대화 (pct 고정)

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value

print("✅ 튜닝 함수 정의 완료 (pct 고정, seed=42, val 샤프 최대화)")

#### 6-3. 튜닝 진행

In [ ]:
import pickle

# 로컬 실행: 세션이 끊겨도 이어할 수 있도록 매 조합 완료 후 체크포인트 저장
CHECKPOINT_PATH = "tuning_results.pkl"

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "rb") as f:
        tuning_results = pickle.load(f)
    print(f"✅ 기존 체크포인트 로드 — 이미 완료된 조합: {list(tuning_results.keys())}")
else:
    tuning_results = {}
    print("✅ 새 tuning_results 딕셔너리 생성 (체크포인트 없음)")


def save_checkpoint():
    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(tuning_results, f)
    print(f"  💾 체크포인트 저장 완료 → {CHECKPOINT_PATH}")

In [59]:
# 1. LGBM × 122열
print("[ 1/6 ] LGBM × 122열 (pct=0.45)")
best_params, best_sharpe = tune_lgbm(X_train, X_val, best_pct=0.45, n_trials=30)
tuning_results["LGBM_122"] = {"params": best_params, "sharpe": best_sharpe}
print(f"  ✅ 완료 → 최고 샤프: {best_sharpe:.4f}")
print(f"  최적 파라미터: {best_params}")

[ 1/6 ] LGBM × 122열 (pct=0.45)


  0%|          | 0/30 [00:00<?, ?it/s]

  ✅ 완료 → 최고 샤프: 0.1917
  최적 파라미터: {'n_estimators': 612, 'max_depth': 6, 'learning_rate': 0.015535948791632313, 'num_leaves': 129, 'subsample': 0.6338970394298867}


In [55]:
# 2. XGB × 122열
if "XGB_122" in tuning_results:
    print("[ 2/6 ] XGB × 122열 — 이미 완료됨, 건너뜀")
else:
    print("[ 2/6 ] XGB × 122열 (pct=0.45)")
    best_params, best_sharpe = tune_xgb(X_train, X_val, best_pct=0.45, n_trials=30)
    tuning_results["XGB_122"] = {"params": best_params, "sharpe": best_sharpe}
    save_checkpoint()
    print(f"  ✅ 완료 → 최고 샤프: {best_sharpe:.4f}")
    print(f"  최적 파라미터: {best_params}")

[ 2/6 ] XGB × 122열 (pct=0.45)


  0%|          | 0/30 [00:00<?, ?it/s]

  💾 체크포인트 저장 완료 → tuning_results.pkl
  ✅ 완료 → 최고 샤프: 0.1930
  최적 파라미터: {'n_estimators': 696, 'max_depth': 5, 'learning_rate': 0.03311829888072381, 'subsample': 0.8186841117373118, 'colsample_bytree': 0.6739417822102108}


In [56]:
# 3. CAT × 122열
if "CAT_122" in tuning_results:
    print("[ 3/6 ] CAT × 122열 — 이미 완료됨, 건너뜀")
else:
    print("[ 3/6 ] CAT × 122열 (pct=0.45)")
    best_params, best_sharpe = tune_cat(X_train, X_val, best_pct=0.45, n_trials=30)
    tuning_results["CAT_122"] = {"params": best_params, "sharpe": best_sharpe}
    save_checkpoint()
    print(f"  ✅ 완료 → 최고 샤프: {best_sharpe:.4f}")
    print(f"  최적 파라미터: {best_params}")

[ 3/6 ] CAT × 122열 (pct=0.45)


  0%|          | 0/30 [00:00<?, ?it/s]

  💾 체크포인트 저장 완료 → tuning_results.pkl
  ✅ 완료 → 최고 샤프: 0.1940
  최적 파라미터: {'iterations': 641, 'depth': 6, 'learning_rate': 0.05458280251572331}


In [60]:
# 4. LGBM × 47열
print("[ 4/6 ] LGBM × 47열 (pct=0.45)")
best_params, best_sharpe = tune_lgbm(X_train_ctrl, X_val_ctrl, best_pct=0.45, n_trials=30)
tuning_results["LGBM_47"] = {"params": best_params, "sharpe": best_sharpe}
print(f"  ✅ 완료 → 최고 샤프: {best_sharpe:.4f}")
print(f"  최적 파라미터: {best_params}")

[ 4/6 ] LGBM × 47열 (pct=0.45)


  0%|          | 0/30 [00:00<?, ?it/s]

  ✅ 완료 → 최고 샤프: 0.1891
  최적 파라미터: {'n_estimators': 807, 'max_depth': 4, 'learning_rate': 0.032676417657817626, 'num_leaves': 97, 'subsample': 0.6185801650879991}


In [58]:
# 5. XGB × 47열
if "XGB_47" in tuning_results:
    print("[ 5/6 ] XGB × 47열 — 이미 완료됨, 건너뜀")
else:
    print("[ 5/6 ] XGB × 47열 (pct=0.45)")
    best_params, best_sharpe = tune_xgb(X_train_ctrl, X_val_ctrl, best_pct=0.45, n_trials=30)
    tuning_results["XGB_47"] = {"params": best_params, "sharpe": best_sharpe}
    save_checkpoint()
    print(f"  ✅ 완료 → 최고 샤프: {best_sharpe:.4f}")
    print(f"  최적 파라미터: {best_params}")

[ 5/6 ] XGB × 47열 (pct=0.45)


  0%|          | 0/30 [00:00<?, ?it/s]

  💾 체크포인트 저장 완료 → tuning_results.pkl
  ✅ 완료 → 최고 샤프: 0.1911
  최적 파라미터: {'n_estimators': 379, 'max_depth': 5, 'learning_rate': 0.04170123411685755, 'subsample': 0.9825308339205491, 'colsample_bytree': 0.6018031829288237}


In [57]:
# 6. CAT × 47열
if "CAT_47" in tuning_results:
    print("[ 6/6 ] CAT × 47열 — 이미 완료됨, 건너뜀")
else:
    print("[ 6/6 ] CAT × 47열 (pct=0.45)")
    best_params, best_sharpe = tune_cat(X_train_ctrl, X_val_ctrl, best_pct=0.45, n_trials=30)
    tuning_results["CAT_47"] = {"params": best_params, "sharpe": best_sharpe}
    save_checkpoint()
    print(f"  ✅ 완료 → 최고 샤프: {best_sharpe:.4f}")
    print(f"  최적 파라미터: {best_params}")

[ 6/6 ] CAT × 47열 (pct=0.45)


  0%|          | 0/30 [00:00<?, ?it/s]

  💾 체크포인트 저장 완료 → tuning_results.pkl
  ✅ 완료 → 최고 샤프: 0.1921
  최적 파라미터: {'iterations': 782, 'depth': 7, 'learning_rate': 0.01947087112740539}


In [ ]:
# 매 조합마다 이미 체크포인트(tuning_results.pkl)로 저장돼 있음 — 최종 확인만
print(f"✅ 저장 완료 ({CHECKPOINT_PATH}) — 현재까지 완료: {list(tuning_results.keys())}")

#### 6-3 완료: 6개 조합 하이퍼파라미터 튜닝 결과
챔피언 모델: CatBoost × 122열 (val 샤프 0.1940, `iterations=641, depth=6, learning_rate=0.05458280251572331`)

 순위 | 모델 | 변수 | 최고 샤프 | 파라미터 |
|---|---|---|---|---|
| 1 | CAT | 122열 | 0.1940 | iterations=641, depth=6, learning_rate=0.0546 |
| 2 | XGB | 122열 | 0.1930 | n_estimators=696, max_depth=5, learning_rate=0.0331, subsample=0.819, colsample_bytree=0.674 |
| 3 | CAT | 47열 | 0.1921 | iterations=782, depth=7, learning_rate=0.0195 |
| 4 | LGBM | 122열 | 0.1917 | n_estimators=612, max_depth=6, learning_rate=0.0155, num_leaves=129, subsample=0.634 |
| 5 | XGB | 47열 | 0.1911 | n_estimators=379, max_depth=5, learning_rate=0.0417, subsample=0.983, colsample_bytree=0.602 |
| 6 | LGBM | 47열 | 0.1891 | n_estimators=807, max_depth=4, learning_rate=0.0327, num_leaves=97, subsample=0.619 |

In [61]:
"""
최종 챔피언 확정용 — tuning_results.pkl의 최적 파라미터로 CAT 재학습
(preds_ctrl/preds_full은 튜닝 전 기본 파라미터라서 이 재학습이 필요함)
"""
from catboost import CatBoostRegressor
import pickle

# ②트리×47 — CAT_47 최적 파라미터
cat47_params = tuning_results["CAT_47"]["params"]
model_cat47 = CatBoostRegressor(**cat47_params, random_state=42, verbose=0)
model_cat47.fit(X_train_ctrl, y_train_excess)
pred_tree47_final = model_cat47.predict(X_val_ctrl)

# ③트리×122 — CAT_122 최적 파라미터
cat122_params = tuning_results["CAT_122"]["params"]
model_cat122 = CatBoostRegressor(**cat122_params, random_state=42, verbose=0)
model_cat122.fit(X_train, y_train_excess)
pred_tree122_final = model_cat122.predict(X_val)

# 검증 — 0.1921, 0.1940 근방 나와야 정상 (tuning_results 저장값과 비교)
print("CAT×47 재현:", sharpe(pred_tree47_final, excess_val, 0.45), "(기대:", tuning_results["CAT_47"]["sharpe"], ")")
print("CAT×122 재현:", sharpe(pred_tree122_final, excess_val, 0.45), "(기대:", tuning_results["CAT_122"]["sharpe"], ")")

# 저장 — 효과분해_최종.ipynb에서 바로 로드하도록
with open("pred_tree47_final.pkl", "wb") as f:
    pickle.dump(pred_tree47_final, f)
with open("pred_tree122_final.pkl", "wb") as f:
    pickle.dump(pred_tree122_final, f)

print("\n저장 완료: pred_tree47_final.pkl, pred_tree122_final.pkl")

CAT×47 재현: 0.1921245102530069 (기대: 0.1921245102530069 )
CAT×122 재현: 0.1940063762222457 (기대: 0.1940063762222457 )

저장 완료: pred_tree47_final.pkl, pred_tree122_final.pkl


---
## 성능 평가
val 데이터까지 포함해 재학습하고 test를 진행한다. <br>
참고: 아래 test 평가(Sharpe 0.1911)는 최초 6:2:2 분할 시의 test(143,595건)로 진행한 공식 test 배부 전 사전 점검이다.보고서 6.4절의 test 결과와는 사용한 데이터셋이 상이하다.

In [ ]:
"""
최종 챔피언 재학습 + test 1회 평가
"""
from catboost import CatBoostRegressor

# 1. train + val 합치기 (X, y 둘 다)
X_trainval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_trainval = pd.concat([y_train_excess, y_val_excess], axis=0).reset_index(drop=True)

print(f"재학습용 데이터: {X_trainval.shape}")  

재학습용 데이터: (574374, 122)


In [71]:
# 2. 최종 파라미터로 챔피언 재학습
champion_final = CatBoostRegressor(
    iterations=641, depth=6, learning_rate=0.05458280251572331,
    random_state=42, verbose=0
)
champion_final.fit(X_trainval, y_trainval)

print("챔피언 재학습 완료")

챔피언 재학습 완료


In [ ]:
# 3. test 세트 준비 
test_mask = (meta['split'] == 'test')
X_test = X_all[test_mask].reset_index(drop=True)
y_test_excess = y_excess[test_mask].reset_index(drop=True)
meta_test = meta[test_mask].reset_index(drop=True)

print(f"X_test shape: {X_test.shape}")  # (143595, 122) 예상

X_test shape: (143595, 122)


In [73]:
# 4. test 예측 (딱 1회)
pred_test = champion_final.predict(X_test)

In [74]:
# 5. 확정된 pct=0.45로 test Sharpe 계산
def sharpe(pred, excess, pct):
    thr = np.quantile(pred, 1 - pct)
    x = np.where(pred >= thr, excess, 0.0)
    sd = x.std(ddof=1)
    return 0.0 if sd == 0 else float(x.mean() / sd)

test_sharpe = sharpe(pred_test, y_test_excess.values, pct=0.45)

print("=" * 50)
print(f"  최종 TEST Sharpe (pct=45%): {test_sharpe:.4f}")
print(f"  벤치마크(A·B등급만): 0.1297")
print(f"  val 결과였던 값: 0.1940")
print("=" * 50)

  최종 TEST Sharpe (pct=45%): 0.1911
  벤치마크(A·B등급만): 0.1297
  val 결과였던 값: 0.1940


In [75]:
# 6. 팀장님 템플릿에 넣을 두 값 계산
thr_final = np.quantile(pred_test, 1 - 0.45)
approve_test = pred_test >= thr_final

x_final = np.where(approve_test, y_test_excess.values, 0.0)
MODEL_SHARPE = x_final.mean() / x_final.std(ddof=1)

approved_excess = y_test_excess.values[approve_test]
MODEL_NEG = (approved_excess < 0).mean() * 100

print(f"MODEL_SHARPE = {MODEL_SHARPE:.4f}")
print(f"MODEL_NEG    = {MODEL_NEG:.2f}")

MODEL_SHARPE = 0.1911
MODEL_NEG    = 8.16


In [76]:
# ══════════ MODEL_SHARPE, MODEL_NEG는 이전 셀에서 이미 계산됨 ══════════
MODEL_NAME = "우리 모델 (상위 45%)"
# ═════════════════════════════════════

import os, sys
import numpy as np
import matplotlib, matplotlib.pyplot as plt

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
FIG = os.path.join(HERE, "figs")
os.makedirs(FIG, exist_ok=True)

matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = ["AppleGothic", "Malgun Gothic", "NanumGothic",
                                           "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["savefig.dpi"] = 300
matplotlib.rcParams["savefig.bbox"] = "tight"
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False

CB, CO, CG, CGRN = "#2C6E9B", "#D9721E", "#8A8A8A", "#3E8E5A"

# test 실측 (전처리팀 산출)
BASE = [
    ("전건승인",             100.0, 0.0853, 14.49, CG),
    ("LC A 등급만",           19.4, 0.1065,  5.25, CG),
    ("LC A·B 등급만",         51.1, 0.1336,  8.64, CG),
    ("LC sub_grade 상위 45%", 45.0, 0.1342,  8.01, CO),
]

if MODEL_SHARPE is None:
    raise SystemExit("MODEL_SHARPE / MODEL_NEG 를 채운 뒤 다시 실행하세요.")

rows = BASE + [(MODEL_NAME, 45.0, MODEL_SHARPE, MODEL_NEG, CGRN)]
names = [r[0] for r in rows]
sh = [r[2] for r in rows]
neg = [r[3] for r in rows]
cc = [r[4] for r in rows]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 5.0))
x = np.arange(len(rows))

a1.bar(x, sh, color=cc, width=0.58)
a1.axhline(0.1342, color=CO, ls="--", lw=1.6)
a1.text(len(rows) - 0.45, 0.1342 + 0.003, "동일 승인률 기준선 0.1342",
        color=CO, fontsize=9.5, ha="right", fontweight="bold")
for i, v in enumerate(sh):
    a1.text(i, v + 0.003, f"{v:.4f}", ha="center", fontsize=10,
            fontweight="bold" if i >= len(rows) - 2 else "normal")
a1.set_xticks(x)
a1.set_xticklabels(names, rotation=18, ha="right", fontsize=9.5)
a1.set_ylabel("포트폴리오 샤프 (동일가중)")
a1.set_ylim(0, max(sh) * 1.22)
a1.set_title("test 샤프 — 거부 건은 초과수익 0(전액 국채)으로 포함", fontsize=11, pad=10)
a1.grid(axis="y", alpha=0.25)

a2.bar(x, neg, color=cc, width=0.58)
for i, v in enumerate(neg):
    a2.text(i, v + 0.2, f"{v:.2f}%", ha="center", fontsize=10,
            fontweight="bold" if i >= len(rows) - 2 else "normal")
a2.set_xticks(x)
a2.set_xticklabels(names, rotation=18, ha="right", fontsize=9.5)
a2.set_ylabel("승인 건 중 excess < 0 비율 (%)")
a2.set_ylim(0, max(neg) * 1.2)
a2.set_title("승인한 대출 중 국채보다 못한 비율 (낮을수록 좋음)", fontsize=11, pad=10)
a2.grid(axis="y", alpha=0.25)

gain = (MODEL_SHARPE / 0.1342 - 1) * 100
fig.suptitle(f"동일 승인률 45% 비교 (test 143,595건) — 모델이 LC 기준선을 {gain:+.1f}% "
             f"{'상회' if gain > 0 else '하회'}", fontsize=13, y=1.04)
fig.text(0.5, -0.10, "승인률 45%는 validation 에서 결정했으며, test 는 챔피언 모형 확정 후 1회만 평가했다.",
         ha="center", fontsize=9.5, color=CG)

p = os.path.join(FIG, "R1_최종_동일승인률비교.png")
fig.savefig(p)
plt.close(fig)
print(f"저장 {p}")
print(f"\n  LC sub_grade 상위 45% : 샤프 0.1342 · excess<0 8.01%")
print(f"  {MODEL_NAME:<22}: 샤프 {MODEL_SHARPE:.4f} · excess<0 {MODEL_NEG:.2f}%")
print(f"  → 샤프 {gain:+.1f}%")

저장 /Users/hyun/-/snu kdt/lending club/modeling/figs/R1_최종_동일승인률비교.png

  LC sub_grade 상위 45% : 샤프 0.1342 · excess<0 8.01%
  우리 모델 (상위 45%)        : 샤프 0.1911 · excess<0 8.16%
  → 샤프 +42.4%


> 결과 해석
- 동일 승인율 45% 기준, 우리 모델(Sharpe 0.1911)이 LC sub_grade 상위 45%(0.1342) 대비 **+42.4%** 높은 포트폴리오 Sharpe를 기록
- 전건승인(0.0853), LC A등급만(0.1065), LC A·B등급만(0.1336) 등 다른 단순 전략도 모두 상회
- 승인건 중 손실(excess<0) 비율은 모델(8.16%)이 LC sub_grade(8.01%)와 거의 비슷한 수준: Sharpe는 크게 개선했지만 손실 비율은 희생하지 않음

In [ ]:
# 챔피언 모델 저장
champion_final.save_model("champion_final.cbm")